# Week 1 — Image basics and I/O

Goals of this notebook:

1. Load a 2D natural image and a 3D NIfTI medical volume in Python.
2. Inspect shape, dtype, and intensity range.
3. Plot the central axial / sagittal / coronal slices of a 3D volume.
4. Plot the **intensity histogram** and confirm that *colormap* and *data* are independent.
5. Apply a 5×5 Gaussian filter and compare PSNR before/after.

Companion reading: `week_1/Readme.md` and `handbook_main.pdf` chapter 'Week 1'.

## Setup

Activate the `mic_soc` environment first. If `nibabel` and `scikit-image` are not yet installed, run:

```bash
pip install nibabel scikit-image
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from skimage import data, filters, metrics
from pathlib import Path

np.random.seed(0)
plt.rcParams['figure.figsize'] = (10, 4)
print('numpy version :', np.__version__)

## 1. A 2D natural image (no medical scanner needed)

scikit-image ships a few sample images. Let's start with `camera` (a classic 8-bit grayscale 512×512).

In [ ]:
img = data.camera()  # uint8, (512, 512)
print('shape :', img.shape)
print('dtype :', img.dtype)
print('min/max :', img.min(), img.max())
print('mean / std :', img.mean(), img.std())

plt.imshow(img, cmap='gray')
plt.title('camera, gray colormap')
plt.axis('off')
plt.show()

### Colormap ≠ data

The same `img` array can be rendered under any colormap. The *data* never changes.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, cmap in zip(axes, ['gray', 'viridis', 'hot', 'inferno']):
    ax.imshow(img, cmap=cmap)
    ax.set_title(f"cmap='{cmap}'")
    ax.axis('off')
plt.tight_layout(); plt.show()

## 2. The intensity histogram

Always look at the histogram alongside the image. Many medical-imaging algorithms (thresholding, histogram equalisation, MAP denoising) depend on it directly.

In [ ]:
plt.hist(img.ravel(), bins=64, color='steelblue')
plt.title('Intensity histogram of `camera`')
plt.xlabel('intensity')
plt.ylabel('pixel count')
plt.show()

## 3. Loading a 3D NIfTI volume

Medical scanners save volumes in formats like NIfTI (`.nii`, `.nii.gz`) or DICOM. We'll use `nibabel` for NIfTI.

**If you have a NIfTI file locally**, put it at `../data/brain.nii.gz` and run the cell below. 
**If not**, the next cell creates a synthetic 3D 'volume' of three concentric ellipsoids so you can still run the notebook end-to-end.

In [ ]:
# Try to load a real volume; fall back to a synthetic phantom.
real_volume_path = Path('../data/brain.nii.gz')
volume = None

if real_volume_path.exists():
    import nibabel as nib
    nii = nib.load(real_volume_path)
    volume = np.asarray(nii.get_fdata())
    print('Loaded NIfTI:', real_volume_path)
    print('shape :', volume.shape, 'dtype:', volume.dtype)
    print('voxel spacing (mm):', nii.header.get_zooms())
else:
    # Synthetic phantom: three nested ellipsoids inside a 96^3 cube.
    N = 96
    z, y, x = np.indices((N, N, N)) - N//2
    r2 = (x/40)**2 + (y/35)**2 + (z/30)**2
    volume = np.zeros((N, N, N), dtype=np.float32)
    volume[r2 < 1.0] = 0.4   # outer
    volume[r2 < 0.5] = 0.7   # mid
    volume[r2 < 0.2] = 1.0   # inner
    print('No NIfTI found; using synthetic phantom.')
    print('shape :', volume.shape, 'dtype:', volume.dtype)

In [ ]:
# Central slice in each of the three axes
z0, y0, x0 = [s // 2 for s in volume.shape]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(volume[z0, :, :], cmap='gray');   axes[0].set_title(f'axial  z={z0}')
axes[1].imshow(volume[:, y0, :], cmap='gray');   axes[1].set_title(f'coronal y={y0}')
axes[2].imshow(volume[:, :, x0], cmap='gray');   axes[2].set_title(f'sagittal x={x0}')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

## 4. Add noise, then denoise with a Gaussian filter

This is your first 'classical' denoising experiment. We compare the PSNR of the noisy image and the filtered image. PSNR (peak signal-to-noise ratio) is a standard reconstruction metric:

$$\mathrm{PSNR}(x, \hat x) = 10 \log_{10}\!\left( \frac{\max(x)^2}{\frac{1}{N}\sum_i (x_i - \hat x_i)^2}\right)$$

In [ ]:
clean = img.astype(np.float32) / 255.0     # in [0, 1]
noisy = clean + np.random.normal(0.0, 0.10, size=clean.shape)
noisy = np.clip(noisy, 0, 1)

denoised = filters.gaussian(noisy, sigma=1.5)

psnr_noisy    = metrics.peak_signal_noise_ratio(clean, noisy,    data_range=1.0)
psnr_denoised = metrics.peak_signal_noise_ratio(clean, denoised, data_range=1.0)
print(f'PSNR noisy    = {psnr_noisy:5.2f} dB')
print(f'PSNR denoised = {psnr_denoised:5.2f} dB  (gain: {psnr_denoised - psnr_noisy:+.2f} dB)')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, im, t in zip(axes, [clean, noisy, denoised], ['clean', 'noisy (σ=0.1)', 'gaussian denoised']):
    ax.imshow(im, cmap='gray', vmin=0, vmax=1); ax.set_title(t); ax.axis('off')
plt.tight_layout(); plt.show()

## 5. Your turn (exercises)

1. Change the noise standard deviation to 0.05, 0.20, 0.40. Plot PSNR vs. noise level.
2. Replace the Gaussian filter with a **mean (box) filter** of size 5×5. Compare visually — which preserves edges better?
3. Compute the 2D Fourier transform of `clean` and of `noisy`. Plot the log-magnitude. Where does the noise show up in the spectrum?
4. Take the brain volume (real or synthetic), pick one slice, and repeat steps 4 above on it.

## What's next

Notebook `02_probability_refresher.ipynb` walks you through the math of **MLE vs MAP** for the mean of a Gaussian, with simulations. This sets up Week 2's Bayesian denoising properly.